In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pulp
from sklearn.model_selection import TimeSeriesSplit

# --- 1. ML FEATURE ENGINEERING & CLASSIFIER ---
def prepare_ml_features(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    df = df.sort_values('timestamp').reset_index(drop=True)

    df['price_lag_24h'] = df['day_ahead_price'].shift(24)
    df['price_lag_48h'] = df['day_ahead_price'].shift(48)
    df['residual_load_lag_24h'] = df['residual_load'].shift(24)

    df['price_rolling_mean_6h'] = df['day_ahead_price'].rolling(window=6).mean()
    df['price_rolling_std_6h'] = df['day_ahead_price'].rolling(window=6).std()
    df['residual_load_rolling_mean_6h'] = df['residual_load'].rolling(window=6).mean()

    df['wind_ramp_1h'] = df['wind_generation'].diff(1)
    df['solar_ramp_1h'] = df['solar_generation'].diff(1)

    df = df.dropna().reset_index(drop=True)
    return df

def define_targets(df: pd.DataFrame, negative_threshold: float = -20.0, spike_quantile: float = 0.90) -> pd.DataFrame:
    positive_threshold = df['day_ahead_price'].quantile(spike_quantile)
    conditions = [
        (df['day_ahead_price'] <= negative_threshold),
        (df['day_ahead_price'] >= positive_threshold)
    ]
    choices = [1, 2]
    df['target_risk_class'] = np.select(conditions, choices, default=0)
    return df

def train_lightgbm_classifier(df: pd.DataFrame):
    feature_cols = [
        'residual_load', 'wind_generation', 'solar_generation',
        'price_lag_24h', 'price_lag_48h', 'residual_load_lag_24h',
        'price_rolling_mean_6h', 'price_rolling_std_6h',
        'residual_load_rolling_mean_6h', 'wind_ramp_1h', 'solar_ramp_1h'
    ]
    X = df[feature_cols]
    y = df['target_risk_class']

    tscv = TimeSeriesSplit(n_splits=3)
    best_model = None

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = lgb.LGBMClassifier(
            objective='multiclass',
            num_class=3,
            n_estimators=150,
            learning_rate=0.05,
            class_weight='balanced',
            random_state=42,
            verbosity=-1
        )
        model.fit(X_train, y_train)
        best_model = model

    return best_model, feature_cols

# --- 2. RISK-AWARE BESS MILP OPTIMIZER ---
def solve_risk_aware_bess_dispatch(
    prices: pd.Series,
    spike_probabilities: pd.Series,
    negative_pricing_probabilities: pd.Series,
    fcr_capacity_price: float,
    max_power: float = 5.0,
    max_energy: float = 10.0,
    eta_charge: float = 0.92,
    eta_discharge: float = 0.92,
    degradation_cost_per_mwh: float = 5.0
) -> pd.DataFrame:

    time_periods = len(prices)
    model = pulp.LpProblem("Risk_Aware_BESS_MultiMarket_Optimization", pulp.LpMaximize)

    P_charge = {t: pulp.LpVariable(f"P_charge_{t}", lowBound=0, upBound=max_power) for t in range(time_periods)}
    P_discharge = {t: pulp.LpVariable(f"P_discharge_{t}", lowBound=0, upBound=max_power) for t in range(time_periods)}
    R_fcr = {t: pulp.LpVariable(f"R_fcr_{t}", lowBound=0, upBound=max_power) for t in range(time_periods)}
    SoC = {t: pulp.LpVariable(f"SoC_{t}", lowBound=0, upBound=max_energy) for t in range(time_periods + 1)}

    u_charge = {t: pulp.LpVariable(f"u_charge_{t}", cat='Binary') for t in range(time_periods)}
    u_discharge = {t: pulp.LpVariable(f"u_discharge_{t}", cat='Binary') for t in range(time_periods)}

    model += SoC[0] == 0.5 * max_energy
    total_objective = 0

    for t in range(time_periods):
        price = prices.iloc[t]
        p_spike = spike_probabilities.iloc[t]
        p_neg = negative_pricing_probabilities.iloc[t]

        da_revenue = price * (P_discharge[t] - P_charge[t])
        fcr_revenue = fcr_capacity_price * R_fcr[t]
        throughput = (P_charge[t] + P_discharge[t]) * 1.0
        deg_cost = degradation_cost_per_mwh * throughput

        risk_bonus_charging = p_neg * abs(price) * P_charge[t] * 0.2
        risk_penalty_premature_discharge = p_spike * max(0, 100 - price) * P_charge[t] * 0.1

        net_period_value = da_revenue + fcr_revenue - deg_cost + risk_bonus_charging - risk_penalty_premature_discharge
        total_objective += net_period_value

    model += total_objective

    for t in range(time_periods):
        model += P_discharge[t] + R_fcr[t] <= max_power
        model += P_charge[t] + R_fcr[t] <= max_power
        model += u_charge[t] + u_discharge[t] <= 1
        model += P_charge[t] <= max_power * u_charge[t]
        model += P_discharge[t] <= max_power * u_discharge[t]

        energy_in = P_charge[t] * eta_charge
        energy_out = P_discharge[t] / eta_discharge
        model += SoC[t + 1] == SoC[t] + energy_in - energy_out

    model += SoC[time_periods] == SoC[0]

    solver = pulp.PULP_CBC_CMD(msg=False)
    model.solve(solver)

    results = []
    for t in range(time_periods):
        results.append({
            "Hour": t,
            "Price_EUR_MWh": prices.iloc[t],
            "Spike_Prob": spike_probabilities.iloc[t],
            "Neg_Prob": negative_pricing_probabilities.iloc[t],
            "P_Charge_MW": pulp.value(P_charge[t]),
            "P_Discharge_MW": pulp.value(P_discharge[t]),
            "R_FCR_MW": pulp.value(R_fcr[t]),
            "SoC_MWh": pulp.value(SoC[t])
        })

    return pd.DataFrame(results)

# --- 3. DETERMINISTIC OPTIMIZER & BACKTEST EVALUATION ---
def run_deterministic_bess_dispatch(
    prices: pd.Series,
    fcr_capacity_price: float,
    max_power: float = 5.0,
    max_energy: float = 10.0,
    eta_charge: float = 0.92,
    eta_discharge: float = 0.92,
    degradation_cost_per_mwh: float = 5.0
) -> pd.DataFrame:
    time_periods = len(prices)
    model = pulp.LpProblem("Deterministic_BESS_Optimization", pulp.LpMaximize)

    P_charge = {t: pulp.LpVariable(f"det_P_charge_{t}", lowBound=0, upBound=max_power) for t in range(time_periods)}
    P_discharge = {t: pulp.LpVariable(f"det_P_discharge_{t}", lowBound=0, upBound=max_power) for t in range(time_periods)}
    R_fcr = {t: pulp.LpVariable(f"det_R_fcr_{t}", lowBound=0, upBound=max_power) for t in range(time_periods)}
    SoC = {t: pulp.LpVariable(f"det_SoC_{t}", lowBound=0, upBound=max_energy) for t in range(time_periods + 1)}

    u_charge = {t: pulp.LpVariable(f"det_u_charge_{t}", cat='Binary') for t in range(time_periods)}
    u_discharge = {t: pulp.LpVariable(f"det_u_discharge_{t}", cat='Binary') for t in range(time_periods)}

    model += SoC[0] == 0.5 * max_energy
    total_objective = 0

    for t in range(time_periods):
        price = prices.iloc[t]
        da_revenue = price * (P_discharge[t] - P_charge[t])
        fcr_revenue = fcr_capacity_price * R_fcr[t]
        throughput = (P_charge[t] + P_discharge[t]) * 1.0
        deg_cost = degradation_cost_per_mwh * throughput
        total_objective += (da_revenue + fcr_revenue - deg_cost)

    model += total_objective

    for t in range(time_periods):
        model += P_discharge[t] + R_fcr[t] <= max_power
        model += P_charge[t] + R_fcr[t] <= max_power
        model += u_charge[t] + u_discharge[t] <= 1
        model += P_charge[t] <= max_power * u_charge[t]
        model += P_discharge[t] <= max_power * u_discharge[t]

        energy_in = P_charge[t] * eta_charge
        energy_out = P_discharge[t] / eta_discharge
        model += SoC[t + 1] == SoC[t] + energy_in - energy_out

    model += SoC[time_periods] == SoC[0]

    solver = pulp.PULP_CBC_CMD(msg=False)
    model.solve(solver)

    results = []
    for t in range(time_periods):
        results.append({
            "Hour": t,
            "Price_EUR_MWh": prices.iloc[t],
            "Det_P_Charge": pulp.value(P_charge[t]),
            "Det_P_Discharge": pulp.value(P_discharge[t]),
            "Det_R_FCR": pulp.value(R_fcr[t]),
            "Det_SoC": pulp.value(SoC[t])
        })

    return pd.DataFrame(results)

def evaluate_financial_performance(risk_aware_df: pd.DataFrame, deterministic_df: pd.DataFrame, fcr_capacity_price: float) -> dict:
    risk_da_rev = (risk_aware_df['Price_EUR_MWh'] * (risk_aware_df['P_Discharge_MW'] - risk_aware_df['P_Charge_MW'])).sum()
    risk_fcr_rev = (fcr_capacity_price * risk_aware_df['R_FCR_MW']).sum()
    risk_throughput = (risk_aware_df['P_Charge_MW'] + risk_aware_df['P_Discharge_MW']).sum()
    risk_deg_cost = risk_throughput * 5.0
    risk_total_net = risk_da_rev + risk_fcr_rev - risk_deg_cost

    det_da_rev = (deterministic_df['Price_EUR_MWh'] * (deterministic_df['Det_P_Discharge'] - deterministic_df['Det_P_Charge'])).sum()
    det_fcr_rev = (fcr_capacity_price * deterministic_df['Det_R_FCR']).sum()
    det_throughput = (deterministic_df['Det_P_Charge'] + deterministic_df['Det_P_Discharge']).sum()
    det_deg_cost = det_throughput * 5.0
    det_total_net = det_da_rev + det_fcr_rev - det_deg_cost

    kpis = {
        "Risk_Aware_Net_Revenue_EUR": round(risk_total_net, 2),
        "Deterministic_Net_Revenue_EUR": round(det_total_net, 2),
        "Revenue_Lift_Percentage": round(((risk_total_net - det_total_net) / abs(det_total_net)) * 100, 2),
        "Risk_Aware_Throughput_MWh": round(risk_throughput, 2),
        "Deterministic_Throughput_MWh": round(det_throughput, 2)
    }
    return kpis

# --- 4. EXECUTION PIPELINE & BACKTEST ---
if __name__ == "__main__":
    print("Step 1: Initializing raw data generation...")
    np.random.seed(42)
    sample_size = 168
    timestamps = pd.date_range(start="2026-09-01", periods=sample_size, freq="h")

    raw_df = pd.DataFrame({
        'timestamp': timestamps,
        'day_ahead_price': np.random.normal(55, 40, sample_size),
        'residual_load': np.random.normal(42000, 6000, sample_size),
        'wind_generation': np.random.uniform(4000, 18000, sample_size),
        'solar_generation': np.maximum(0, np.random.normal(3500, 4500, sample_size))
    })

    print("Step 2: Preparing ML features and defining risk targets...")
    processed_df = prepare_ml_features(raw_df)
    labeled_df = define_targets(processed_df)

    print("Step 3: Training LightGBM risk classifier...")
    trained_model, feature_cols = train_lightgbm_classifier(labeled_df)

    probabilities = trained_model.predict_proba(labeled_df[feature_cols])
    neg_pricing_probs = pd.Series(probabilities[:, 1])
    spike_probs = pd.Series(probabilities[:, 2])
    prices_subset = labeled_df['day_ahead_price'].reset_index(drop=True)

    print("Step 4: Executing Risk-Aware MILP BESS Optimization...")
    optimization_results = solve_risk_aware_bess_dispatch(
        prices=prices_subset,
        spike_probabilities=spike_probs,
        negative_pricing_probabilities=neg_pricing_probs,
        fcr_capacity_price=14.0
    )

    print("Step 5: Running deterministic backtest & evaluating financial KPIs...")
    det_results = run_deterministic_bess_dispatch(prices=prices_subset, fcr_capacity_price=14.0)
    kpis = evaluate_financial_performance(optimization_results, det_results, fcr_capacity_price=14.0)

    print("\n--- Pipeline Execution & Financial Backtest Complete ---")
    for metric, val in kpis.items():
        print(f"{metric}: {val}")

Step 1: Initializing raw data generation...
Step 2: Preparing ML features and defining risk targets...
Step 3: Training LightGBM risk classifier...
Step 4: Executing Risk-Aware MILP BESS Optimization...
Step 5: Running deterministic backtest & evaluating financial KPIs...

--- Pipeline Execution & Financial Backtest Complete ---
Risk_Aware_Net_Revenue_EUR: 15149.7
Deterministic_Net_Revenue_EUR: 15150.89
Revenue_Lift_Percentage: -0.01
Risk_Aware_Throughput_MWh: 264.15
Deterministic_Throughput_MWh: 264.15
